# Artist & Collaboration Feature Enrichment

This notebook adds three artist-level features to the audio-feature dataset produced in
`02_extract_audio_features.ipynb`:

1. **Collaboration count** — number of credited artists per track (and a binary `collaboration_type` flag).
2. **Monthly listeners** — scraped from [kworb.net](https://kworb.net/spotify/listeners.html) for the primary (first-listed) artist.
3. **Popularity score** — scraped from Groover's public Spotify popularity endpoint for every unique credited artist.

Output: `data/processed/processed_chart_tracks_enriched_features.csv`.

In [ ]:
import pandas as pd
import numpy as np
import lxml
import requests

In [2]:
df = pd.read_csv("../data/processed/processed_chart_tracks_audio_features.csv")

In [ ]:
def count_artists(artist_names):
    """Count the number of artists in the artist_name column."""
    if pd.isna(artist_names):
        return 0
    
    if "," in artist_names:
        artists = artist_names.strip('"').split(',')
        return len(artists)

    else : 
        return 1 #single artist


df['artist_count'] = df['artist_names'].apply(count_artists)
df['collaboration_type'] = np.where(df['artist_count'] > 1, 1, 0) # 1 is collab, 0 is solo

In [ ]:
def get_kworb_listeners():
    """Scrape the full listeners table from kworb into a dict {artist: listeners}."""
    all_data = {}
    for i in range(1, 6):  
        url = "https://kworb.net/spotify/listeners.html" if i == 1 else f"https://kworb.net/spotify/listeners{i}.html"
        try:
            tables = pd.read_html(url)
            df = tables[0][["Artist", "Listeners"]]
            df["Listeners"] = df["Listeners"].astype(str).str.replace(",", "").astype(int)
            for _, row in df.iterrows():
                all_data[row["Artist"].lower()] = row["Listeners"]
        except Exception:
            break
    return all_data

def lookup_listeners(artist_name, listeners_dict):
    return listeners_dict.get(artist_name.strip('"').lower(), None)

listeners_dict = get_kworb_listeners()

df["monthly_listeners"] = df["artist_names"].apply(
    lambda x: lookup_listeners(x.strip('"').split(",")[0].strip(), listeners_dict) #take the first/ primary artist
)

### Artist popularity score (Groover)

We scrape the **Spotify popularity score** for every unique credited artist from Groover's public endpoint
(`https://groover.co/core/distantapi/spotify/popularity/`) — the same number that powers the
[Spotify Popularity Score tool](https://groover.co/en/lp/free-tools/spotify-popularity-score/).

Notes on the scraper below:

- We use only the standard library (`urllib`, `csv`, `concurrent.futures`) — no extra dependencies.
- A small `ThreadPoolExecutor` (5 workers) speeds things up while staying polite.
- Each artist is retried up to 3 times with linear backoff.
- The script is **resumable**: it reads the existing `artist_popularity_data.csv` and only re-fetches artists whose `popularity_score` is still `None`, so we don't re-hit the endpoint for artists we've already scored.
- Output: `artist_popularity_data.csv` with columns `artist_name`, `popularity_score`. We commit this file to the repo as `data/raw/raw_artist_popularity_scores.csv`, and the merge step further down loads it back in to attach popularity to every row of the chart dataframe.

In [ ]:
import csv
import json
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

INPUT_CSV = "final_songs_with_audio_features.csv"
OUTPUT_CSV = "artist_popularity_data.csv"
ARTIST_COLUMN = "artist_names"
MAX_WORKERS = 5
REQUEST_RETRIES = 3
RETRY_DELAY_SECONDS = 1.0


def extract_unique_artists(csv_path, artist_column):
    unique_artists = []
    seen = set()

    with open(csv_path, newline="", encoding="utf-8") as csv_file:
        reader = csv.DictReader(csv_file)

        if artist_column not in reader.fieldnames:
            raise ValueError(f"Column '{artist_column}' was not found in {csv_path}")

        for row in reader:
            artist_value = row.get(artist_column, "")
            for artist in artist_value.split(","):
                cleaned_artist = artist.strip()
                if cleaned_artist and cleaned_artist not in seen:
                    seen.add(cleaned_artist)
                    unique_artists.append(cleaned_artist)

    return unique_artists


def read_existing_results(output_csv):
    results = {}

    output_path = Path(output_csv)
    if not output_path.exists():
        return results

    with open(output_csv, newline="", encoding="utf-8") as csv_file:
        for row in csv.DictReader(csv_file):
            score = row.get("popularity_score", "").strip()
            results[row["artist_name"]] = {
                "artist_name": row["artist_name"],
                "popularity_score": int(score) if score else None,
            }

    return results


def fetch_artist_popularity(artist):
    base_url = "https://groover.co/core/distantapi/spotify/popularity/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Referer": "https://groover.co/en/lp/free-tools/spotify-popularity-score/",
        "Accept": "application/json"
    }

    for attempt in range(1, REQUEST_RETRIES + 1):
        try:
            query_url = f"{base_url}?{urlencode({'q': artist})}"
            request = Request(query_url, headers=headers)

            with urlopen(request, timeout=15) as response:
                data = json.load(response)
                if isinstance(data, str):
                    data = json.loads(data)

            score = data.get("popularity")
            if score is not None:
                return {
                    "artist_name": artist,
                    "popularity_score": score,
                }
        except Exception as e:
            if attempt == REQUEST_RETRIES:
                print(f"Searching: {artist}... ERROR: {e}")

        time.sleep(RETRY_DELAY_SECONDS * attempt)

    return {"artist_name": artist, "popularity_score": None}


def scrape_groover_batch(artist_list):
    print(f"--- Processing {len(artist_list)} Artists ---")

    all_artist_data = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        for entry in executor.map(fetch_artist_popularity, artist_list):
            print(
                f"Searching: {entry['artist_name']}... DONE (Score: {entry['popularity_score']})"
            )
            all_artist_data.append(entry)
            time.sleep(0.1)

    return all_artist_data


def main():
    input_path = Path(INPUT_CSV)
    if not input_path.exists():
        raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")

    unique_artists = extract_unique_artists(INPUT_CSV, ARTIST_COLUMN)
    print(f"Found {len(unique_artists)} unique artists in {INPUT_CSV}.")
    existing_results = read_existing_results(OUTPUT_CSV)
    artists_to_fetch = [
        artist
        for artist in unique_artists
        if existing_results.get(artist, {}).get("popularity_score") is None
    ]

    print(f"Retrying {len(artists_to_fetch)} artists with missing popularity scores.")

    fetched_results = scrape_groover_batch(artists_to_fetch)
    for entry in fetched_results:
        existing_results[entry["artist_name"]] = entry

    results = [existing_results[artist] for artist in unique_artists]

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=["artist_name", "popularity_score"])
        writer.writeheader()
        writer.writerows(results)

    print(f"\nSaved artist popularity scores to {OUTPUT_CSV}")
    print(results[:5])


if __name__ == "__main__":
    main()

In [ ]:
# find primary artist's popularity score
popularity_df = pd.read_csv("../data/raw/raw_artist_popularity_scores.csv")

def find_primary_artist(artist_names):
    """find primary artist (first listed)"""
    if "," in artist_names:
        return artist_names.strip('"').split(",")[0].strip()
    return artist_names

df["primary_artist"] = df["artist_names"].apply(find_primary_artist)

merged_df = (
    df.merge(popularity_df, left_on="primary_artist", right_on="artist_name", how="left")
      .drop(columns=["artist_name", "primary_artist"])
      .reset_index(drop=True)
)

In [ ]:
merged_df.to_csv("../data/processed/processed_chart_tracks_enriched_features.csv", index=False)

In [ ]:
#check
new_df = pd.read_csv("../data/processed/processed_chart_tracks_enriched_features.csv")
print("Columns:", list(new_df.columns))
print(f"\nRows: {len(new_df):,}")
print("\nMissing values per column:")
new_df.isna().sum()